[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZacharyHillman18/PharmapyWorkshop/blob/main/Workshop_Notebooks/FlowsheetOptimization.ipynb)

# Flowsheet optimization: batch, semibatch or continuous?

In this notebook we build complete **reaction + crystallization** flowsheets with PharmaPy and choose the best one. This is a **discrete (superstructure) optimization**: each unit can be run in batch, semibatch or continuous mode, and continuous reactors can be staged in series.

### Chemistry
The desired product **C** competes with a consecutive side reaction:

$$ A + B \xrightarrow{r_1} C \qquad\qquad B + C \xrightarrow{r_2} D $$

$$ r_1 = k_1\, C_A C_B, \qquad r_2 = k_2\, C_B^2\, C_C, \qquad k_i = k_{i, ref} \exp\left[-\frac{E_i}{R}\left(\frac{1}{T} - \frac{1}{T_{ref}}\right)\right] $$

The side reaction is **second order in B** and has a **higher activation energy**. Keeping the concentration of B low (dosing it slowly) and working at lower temperatures favors C, but slows the process down.

Molecular weights are A = 100, B = 100, C = 200 and D = 300 g/mol, so both reactions close the mass balance. **C crystallizes** (with the kinetics of the crystallizer notebook), while A, B and D stay in solution.

### Superstructure

| Section | Options |
|---|---|
| Reaction | `batch`; `semibatch` (A charged, B dosed, then a batch hold); `continuous`, `continuous x2`, `continuous x3` (CSTRs in series) |
| Crystallization | `batch` (cooling); `semibatch` (cold vessel that collects the reactor effluent); `continuous` (MSMPR) |

Not every combination is possible: a batch product can only be charged to a batch crystallizer, and a continuous stream needs a continuous or semibatch receiver. Going from batch to continuous (or vice versa) would need a buffer tank, which is not part of this exercise.

### Key performance indicators (KPIs)
Computed from the contents of the last crystallizer (liquid + crystals), in moles:

$$ \text{Selectivity} = \frac{n_C}{n_C + n_D}, \qquad \text{Yield} = \frac{n_{C,\,crystals}}{n_B + n_C + 2 n_D} $$

The yield denominator is the amount of B that was fed. Every C contains one B and every D two, so we don't need to track the feeds explicitly (for continuous units at steady state, holdup composition = outlet composition).

The crystallizers solve the full crystal size distribution (1D-FVM), so the product quality is described by the volume-weighted median size $d_{50}$ and the **span** $= (d_{90} - d_{10})/d_{50}$. The objective to **maximize** is a weighted sum:

$$ J = w_Y\, \text{Yield} + w_S\, \text{Selectivity} + w_L \frac{d_{50}}{400\ \mu m} - w_W\, \mathrm{span} $$

The reference size puts the terms on comparable scales (all of order one). Choosing it is itself a weighting decision.

**Running on Google Colab?** Run the next cell first. It installs PharmaPy and puts this notebook's data files and `workshop_utils.py` on the Colab machine (about half a minute). Run it again whenever the runtime restarts. To keep your work, use *File → Save a copy in Drive*. On your own computer the cell skips the install and only unpacks the files.

In [ ]:
# @title Colab setup: run this cell first { display-mode: "form" }
# Installs PharmaPy and unpacks workshop_utils.py and data/ into the working
# directory. Run it again after a runtime restart or a factory reset.
import base64, importlib.metadata, importlib.util, io, os, subprocess, sys, zipfile

if 'google.colab' in sys.modules or importlib.util.find_spec('PharmaPy') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pharmapy-org>=0.3.1'], check=True)

_FILES = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_FILES))) as archive:
    archive.extractall('.')
importlib.invalidate_caches()

try:
    _version = importlib.metadata.version('pharmapy-org')
except importlib.metadata.PackageNotFoundError:   # a development install of PharmaPy
    _version = '(development install)'
print('PharmaPy', _version, '- workshop files ready in', os.getcwd())

In [ ]:
import json
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from PharmaPy.Phases_Refactored import LiquidPhase, SolidPhase
from PharmaPy.Streams_Refactored import LiquidStream
from PharmaPy.Reactors_Refactored import BatchReactor, SemiBatchReactor, ContinuousReactor
from PharmaPy.Crystallizers_Refactored import BatchCrystallizer, SemiBatchCrystallizer, ContinuousCrystallizer
from PharmaPy.Mechanisms import OneDFVMMechanism
from PharmaPy.Kinetics import RxnKinetics, CrystKinetics
from PharmaPy.ProcessControl_Refactored import SimpleTemperatureController, ContinuousVesselController
from PharmaPy.IntegratorBackends import ScipyBackend
from PharmaPy.SimExec import SimulationExec

from workshop_utils import apply_pharmapy_patches, csd_quantiles, volume_density

apply_pharmapy_patches()

path = 'data/compounds_workshop.json'
with open(path) as f:
    database = json.load(f)

SPECIES = list(database.keys())
MW = np.array([database[name]['mw'] for name in SPECIES], dtype=float)
RHO_SOLVENT = database['solvent']['rho_liq']
print(dict(zip(SPECIES, MW)))

# Size grid of the crystallizers. Coarser than in the crystallizer notebook (60 cells)
# to keep the flowsheets fast; d50 and span are therefore a bit less accurate.
X_GRID = np.geomspace(1, 1500, 35)   # um
SEED_SIZE = 100.0                    # um


def integrator(maxh=600):
    """A new solver for each unit. ScipyBackend only needs scipy (no Assimulo).
    stall_calls: give up after 20000 model calls without progress (fail fast)."""
    return ScipyBackend(options={'maxh': maxh}, stall_calls=20000)

## Kinetics

`params_f` sets the reaction orders explicitly: one list per reaction, with the orders of its reactants (in the order of the json file). Reaction 2 is second order in B and first order in C.

In [ ]:
T_REF = 333.15
RXN_ORDERS = ...  # <-- COMPLETE THIS: reaction orders of the reactants: r1 = k1 CA CB, r2 = k2 CB^2 CC


def rxn_kinetics():
    return RxnKinetics(path, rxn_list=['A + B --> C', 'B + C --> D'],
                       k_params=[5e-3, 2e-3],     # L/(mol s) and L^2/(mol^2 s) at T_REF
                       ea_params=[40e3, 70e3],    # J/mol
                       temp_ref=T_REF,
                       params_f=RXN_ORDERS)


def cryst_kinetics():   # same as in the crystallizer notebook
    return CrystKinetics(np.array([-28.13909202, 0.001, 5.900800253]),
                         nucl_prim=(1.0448e17, 0, 8.30671806),     # #/(m3 s)
                         growth=(2.4297e4, 0, 4.52241037),         # um/s
                         solubility_type='apelblat', solubility_basis='mass_per_volume_solvent')


# Mass balance check: every reaction must conserve mass
kin = rxn_kinetics()
print('stoichiometric matrix (A, B, C, D):\n', kin.stoich_matrix)
mass_change = ...  # <-- COMPLETE THIS: stoichiometric matrix times the molecular weights of A, B, C, D
print('mass change per reaction (g/mol):', mass_change)

## Operating conditions (base case)

All flowsheets process the same feed (A and B at 1.4 mol/L each) and use the same time budget. They differ only in the way the units are operated.

In [ ]:
OPS = {
    'C0': 1.4,         # mol/L of A and of B in the feed (equimolar)
    'T_R': 333.15,     # reactor temperature (K)
    't_R': 3600.,      # reaction time: batch time, total CSTR residence time, or dosing + hold (s)
    'dose_frac': 0.75, # semibatch: fraction of t_R used to dose B (the rest is a batch hold)
    'V_R': 1e-3,       # total reactor volume (m3)
    'T_C': 273.15,     # final / last crystallizer temperature (K)
    'T_C1': 293.15,    # first MSMPR temperature when two MSMPRs are used (K)
    't_C': 3600.,      # batch cooling time, or total MSMPR residence time (s)
    'cool_exp': 1.0,   # batch cooling profile exponent (1 = linear)
    'seed_load': 0.0,  # batch crystallizer seeds, % of the C that can be formed
    'n_tau': 6,        # continuous flowsheets: simulated time in total residence times
}

## Unit operation "factories"

Each function returns a **new**, fully specified unit (units and integrators cannot be reused between simulations). Together they cover the six unit types used in this notebook. Note the common pattern:

1. unit with its own `integrator` and a temperature `controller`
2. `unit.Phases = ...` (for crystallizers: `[liquid, solid]`, with the population balance attached to the solid)
3. `unit.RxnKinetics = ...` or `unit.CrystKinetics = ...`
4. `unit.Inlet = LiquidStream(...)` for units with a feed (only the *first* unit of a flowsheet; `SimulationExec` connects the others)

Downstream units get a placeholder `Phases`; `SimulationExec` replaces the liquid with the upstream material.

In [ ]:
def const_temp(T):
    return lambda t: T


def solvent_liquid(T, vol, mole_conc=(0, 0, 0, 0, 0)):
    return LiquidPhase(path, temp=T, vol=vol, mole_conc=list(mole_conc), name_solv='solvent')


def saturated_liquid(T, vol):
    c_sat = cryst_kinetics().get_solubility(T)
    w_sat = c_sat / (c_sat + RHO_SOLVENT)
    return LiquidPhase(path, temp=T, vol=vol, mass_frac=[0, 0, w_sat, 0, 1 - w_sat], name_solv='solvent')


def fvm_solid(T, seed_mass=0.0, vol_slurry=None):
    """Solid phase of C with a 1D-FVM population balance: empty, or seeded with seed_mass kg of 100 um seeds."""
    solid = SolidPhase(path, temp=T, mass=0, mass_frac=[0, 0, 1, 0, 0])
    seed_shape = np.exp(-0.5 * (np.log(X_GRID / SEED_SIZE) / 0.15)**2)   # volume-fraction shape on X_GRID
    if seed_mass > 0:
        solid.mechanisms = OneDFVMMechanism(solid, target_components='C', solvent_name='solvent',
                                            x_grid=X_GRID, distrib_init=seed_shape, distrib_basis='vol_perc',
                                            basis_mass=seed_mass, vol_slurry=vol_slurry)
    else:
        solid.mechanisms = OneDFVMMechanism(solid, target_components='C', solvent_name='solvent',
                                            x_grid=X_GRID, distrib_init=np.zeros_like(X_GRID))
    return solid


# ------------------------------------------------------------------ reactors
def make_batch_reactor(ops, liquid):
    reactor = BatchReactor(integrator=integrator(), controller=SimpleTemperatureController(temp_func=const_temp(ops['T_R'])))
    reactor.Phases = liquid
    reactor.RxnKinetics = rxn_kinetics()
    return reactor


def make_semibatch_reactor(ops):
    """Half of the volume is charged with A, the other half (B) is dosed."""
    t_dose = ops['dose_frac'] * ops['t_R']
    reactor = SemiBatchReactor(integrator=integrator(), controller=SimpleTemperatureController(temp_func=const_temp(ops['T_R'])))
    reactor.Phases = solvent_liquid(ops['T_R'], ops['V_R'] / 2, (2 * ops['C0'], 0, 0, 0, 0))
    reactor.RxnKinetics = rxn_kinetics()
    # <-- COMPLETE THIS: feed B (2*C0 mol/L, half of V_R dosed over t_dose) through reactor.Inlet
    ...
    return reactor


def make_cstr(ops, vol, feed=None):
    # <-- COMPLETE THIS: ContinuousReactor with its own integrator() and a ContinuousVesselController at T_R
    ...
    reactor.Phases = solvent_liquid(ops['T_R'], vol)
    reactor.RxnKinetics = rxn_kinetics()
    if feed is not None:
        reactor.Inlet = feed
    return reactor


# ------------------------------------------------------------- crystallizers
def make_batch_cryst(ops):
    """Cools from T_R to T_C in t_C, then holds for 30 min. Optional seeds (100 um)."""
    T_R, T_C, t_C, p = ops['T_R'], ops['T_C'], ops['t_C'], ops['cool_exp']

    def temp_func(t):
        return T_C + (T_R - T_C) * (1 - min(t / t_C, 1.0)**p)

    seed_mass = ops['seed_load'] / 100 * ops['C0'] * ops['V_R'] * MW[2]   # kg

    cryst = BatchCrystallizer(integrator=integrator(60), controller=SimpleTemperatureController(temp_func=temp_func))
    cryst.Phases = [solvent_liquid(T_R, ops['V_R']), fvm_solid(T_R, seed_mass, ops['V_R'])]
    cryst.CrystKinetics = cryst_kinetics()
    cryst.seed_mass = seed_mass
    return cryst


def make_semibatch_cryst(ops):
    """Cold vessel with a small saturated heel that collects the reactor effluent."""
    cryst = SemiBatchCrystallizer(integrator=integrator(), controller=SimpleTemperatureController(temp_func=const_temp(ops['T_C'])))
    cryst.Phases = [saturated_liquid(ops['T_C'], 0.1 * ops['V_R']), fvm_solid(ops['T_C'])]
    cryst.CrystKinetics = cryst_kinetics()
    return cryst


def make_msmpr(ops, T, vol):
    cryst = ContinuousCrystallizer(integrator=integrator(), controller=ContinuousVesselController(temp_func=const_temp(T)))
    # <-- COMPLETE THIS: start-up full of saturated solution at T, with an empty FVM solid; then attach cryst_kinetics()
    ...
    return cryst

## Example 1: batch reactor → batch crystallizer

Units are connected with `SimulationExec` and a flowsheet string. At the end of the reaction, the liquid in `R01` is transferred to `CR01`. Each unit gets its own solver settings through `kwargs_run`.

In [ ]:
tic = time.perf_counter()

flst = ...  # <-- COMPLETE THIS: SimulationExec connecting R01 to CR01
flst.R01 = make_batch_reactor(OPS, solvent_liquid(OPS['T_R'], OPS['V_R'], (OPS['C0'], OPS['C0'], 0, 0, 0)))
flst.CR01 = make_batch_cryst(OPS)

run_kwargs = {'R01': {'time_grid': np.linspace(0, OPS['t_R'], 31), 'verbose': False},
              'CR01': {'time_grid': np.linspace(0, OPS['t_C'] + 1800, 41), 'verbose': False}}
flst.SolveFlowsheet(kwargs_run=run_kwargs, verbose=False)

print('simulation time: %.1f s' % (time.perf_counter() - tic))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
res_R = flst.R01.result
axes[0].plot(res_R.time / 60, res_R.mole_conc_liquid0[:, :4])
axes[0].legend(SPECIES[:4])
axes[0].set_ylabel('$C_j$ (mol/L)')
axes[0].set_xlabel('time (min)')
axes[0].set_title('R01 (batch reactor)')

res_C = flst.CR01.result
axes[1].plot(res_C.time / 60, res_C.global_temp)
axes[1].set_ylabel('T (K)')
axes[1].set_xlabel('time (min)')
axes[1].set_title('CR01 (batch crystallizer)')
axes[2].semilogx(*volume_density(flst.CR01))
axes[2].set_ylabel(r'volume density $q_3(\ln L)$')
axes[2].set_xlabel(r'size ($\mu$m)')
axes[2].set_title('final CSD')
fig.tight_layout()
plt.show()

### KPIs
The function below computes the KPIs from the last unit of any flowsheet. `flst.execution_names` lists the units in execution order. The crystals are described by their population balance (the *mechanism* of the solid phase); `csd_quantiles` turns its final distribution into $d_{10}$, $d_{50}$, $d_{90}$ and the span.

In [ ]:
def flowsheet_kpis(flst):
    last = getattr(flst, flst.execution_names[-1])
    res = last.result

    n_liq = res.mass_j_liquid0[-1] / MW
    crystal_mass = last.Phases.Solids[0].mass - getattr(last, 'seed_mass', 0.0)
    n_cryst = crystal_mass / MW[2]                                        # kmol of C formed as crystals
    n_C = n_liq[2] + n_cryst

    selectivity = ...  # <-- COMPLETE THIS: moles of C over moles of C + D
    yield_B = ...  # <-- COMPLETE THIS: crystallized C over B fed (B + C + 2 D)

    mech = last.Phases.Solids[0].mechanisms[0]
    csd = ...  # <-- COMPLETE THIS: d10, d50, d90 and span of the final CSD

    return {'selectivity': selectivity, 'yield': yield_B, 'd50': csd['d50'], 'span': csd['span']}


print({key: round(val, 3) for key, val in flowsheet_kpis(flst).items()})

## Example 2: CSTR → MSMPR (continuous)

For the continuous flowsheet only the first unit has an `Inlet`. The units start up empty (solvent / saturated solution) and are simulated for `n_tau` total residence times to reach steady state. `SimulationExec` passes the whole time-dependent outlet of the CSTR to the MSMPR.

In [ ]:
vol_flow = OPS['V_R'] / OPS['t_R']
feed = LiquidStream(path, temp=OPS['T_R'], vol_flow=vol_flow,
                    mole_conc=[OPS['C0'], OPS['C0'], 0, 0, 0], name_solv='solvent')

tic = time.perf_counter()
flst_cont = SimulationExec(path, flowsheet='R01 --> CR01')
flst_cont.R01 = make_cstr(OPS, OPS['V_R'], feed)
flst_cont.CR01 = make_msmpr(OPS, OPS['T_C'], vol_flow * OPS['t_C'])

time_grid = np.linspace(0, OPS['n_tau'] * (OPS['t_R'] + OPS['t_C']), 41)
# <-- COMPLETE THIS: solve the flowsheet with the same time_grid for both units (hint: kwargs_run={'R01': {...}, 'CR01': {...}})
...
print('simulation time: %.1f s' % (time.perf_counter() - tic))
print({key: round(val, 3) for key, val in flowsheet_kpis(flst_cont).items()})

res_C = flst_cont.CR01.result
mech = flst_cont.CR01.Phases.Solids[0].mechanisms[0]
d50_t = [csd_quantiles(mech.x_grid, distrib, mech.dx)['d50'] for distrib in res_C.distrib_solid0]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(flst_cont.R01.result.time / 3600, flst_cont.R01.result.mole_conc_liquid0[:, :4])
axes[0].legend(SPECIES[:4])
axes[0].set_ylabel('CSTR $C_j$ (mol/L)')
axes[1].plot(res_C.time / 3600, d50_t)
axes[1].set_ylabel(r'MSMPR $d_{50}$ ($\mu$m)')
for ax in axes[:2]:
    ax.set_xlabel('time (h)')
axes[2].semilogx(*volume_density(flst.CR01), label='batch (Example 1)')
axes[2].semilogx(*volume_density(flst_cont.CR01), label='MSMPR (steady state)')
axes[2].set_xlabel(r'size ($\mu$m)')
axes[2].set_ylabel(r'volume density $q_3(\ln L)$')
axes[2].legend()
fig.tight_layout()
plt.show()

## A general flowsheet builder

`build_flowsheet` assembles any feasible combination of the superstructure options. Unit names are generated automatically (`R01`, `R02`, ..., `CR01`, `CR02`, ...), and the flowsheet string joins them with `' --> '`.

In [ ]:
REACTOR_OPTIONS = ['batch', 'semibatch', 'continuous', 'continuous x2', 'continuous x3']
CRYST_OPTIONS = ['batch', 'semibatch', 'continuous']   # 'continuous x2' also works (about 30 s per flowsheet)


def parse(option):
    kind, _, number = option.partition(' x')
    return kind, int(number) if number else 1


def is_feasible(reactor_option, cryst_option):
    reactor_kind, _ = parse(reactor_option)
    cryst_kind, _ = parse(cryst_option)
    # <-- COMPLETE THIS: batch/semibatch reactors can only feed a batch crystallizer; continuous reactors need a continuous or semibatch crystallizer
    ...


def build_flowsheet(reactor_option, cryst_option, ops):
    reactor_kind, n_reactors = parse(reactor_option)
    cryst_kind, n_cryst = parse(cryst_option)
    units, run_kwargs = {}, {}

    # ---- reaction section
    if reactor_kind == 'batch':
        units['R01'] = make_batch_reactor(ops, solvent_liquid(ops['T_R'], ops['V_R'], (ops['C0'], ops['C0'], 0, 0, 0)))
        run_kwargs['R01'] = {'time_grid': np.linspace(0, ops['t_R'], 31)}
    elif reactor_kind == 'semibatch':
        t_dose = ops['dose_frac'] * ops['t_R']
        units['R01'] = make_semibatch_reactor(ops)
        units['R02'] = make_batch_reactor(ops, solvent_liquid(ops['T_R'], ops['V_R']))
        run_kwargs['R01'] = {'time_grid': np.linspace(0, t_dose, 31)}
        run_kwargs['R02'] = {'time_grid': np.linspace(0, ops['t_R'] - t_dose, 16)}
    else:
        feed = LiquidStream(path, temp=ops['T_R'], vol_flow=ops['V_R'] / ops['t_R'],
                            mole_conc=[ops['C0'], ops['C0'], 0, 0, 0], name_solv='solvent')
        for ind in range(n_reactors):
            units['R%02d' % (ind + 1)] = make_cstr(ops, ops['V_R'] / n_reactors, feed if ind == 0 else None)

    # ---- crystallization section
    if cryst_kind == 'batch':
        units['CR01'] = make_batch_cryst(ops)
        run_kwargs['CR01'] = {'time_grid': np.linspace(0, ops['t_C'] + 1800, 41)}
    elif cryst_kind == 'semibatch':
        units['CR01'] = make_semibatch_cryst(ops)
    else:
        vol_C = ops['V_R'] / ops['t_R'] * ops['t_C']
        temps = [ops['T_C']] if n_cryst == 1 else np.linspace(ops['T_C1'], ops['T_C'], n_cryst)
        for ind, T in enumerate(temps):
            units['CR%02d' % (ind + 1)] = make_msmpr(ops, T, vol_C / n_cryst)

    # ---- continuous flowsheets: all units share one time horizon
    if reactor_kind == 'continuous':
        horizon = ops['n_tau'] * ops['t_R'] if cryst_kind == 'semibatch' else ops['n_tau'] * (ops['t_R'] + ops['t_C'])
        run_kwargs = {name: {'time_grid': np.linspace(0, horizon, 41)} for name in units}

    flst = ...  # <-- COMPLETE THIS: connect all the units in series
    for name, unit in units.items():
        setattr(flst, name, unit)
    for kwargs in run_kwargs.values():
        kwargs['verbose'] = False

    return flst, run_kwargs


def evaluate(reactor_option, cryst_option, ops=OPS):
    flst, run_kwargs = build_flowsheet(reactor_option, cryst_option, ops)
    flst.SolveFlowsheet(kwargs_run=run_kwargs, verbose=False)
    return flowsheet_kpis(flst)

## Discrete optimization: enumerate the superstructure

The superstructure is small, so we can simply evaluate **every feasible combination** (under a minute). This is the most reliable way to solve a small discrete problem, and it also shows *how much* worse the alternatives are.

In [ ]:
tic = time.perf_counter()
rows = []
for reactor_option, cryst_option in itertools.product(REACTOR_OPTIONS, CRYST_OPTIONS):
    if not is_feasible(reactor_option, cryst_option):
        continue

    tic_case = time.perf_counter()
    kpis = evaluate(reactor_option, cryst_option)
    rows.append({'reaction': reactor_option, 'crystallization': cryst_option, **kpis,
                 'cpu (s)': time.perf_counter() - tic_case})
    print('%-15s -> %-14s done (%.1f s)' % (reactor_option, cryst_option, rows[-1]['cpu (s)']))

results = pd.DataFrame(rows)
print('\ntotal time: %.0f s' % (time.perf_counter() - tic))

In [ ]:
W_YIELD, W_SEL, W_SIZE, W_SPAN = 1.0, 1.0, 1.0, 1.0
SIZE_REF = 400.0  # um


def objective(kpis, w_yield=W_YIELD, w_sel=W_SEL, w_size=W_SIZE, w_span=W_SPAN):
    J = ...  # <-- COMPLETE THIS: weighted-sum objective (to maximize)
    return J


results['J'] = results.apply(objective, axis=1)
ranked = results.sort_values('J', ascending=False).reset_index(drop=True)
print(ranked.round(3).to_string())

In [ ]:
labels = ranked['reaction'] + '\n→ ' + ranked['crystallization']
columns = [('selectivity', 'selectivity (-)'), ('yield', 'yield on B (-)'),
           ('d50', '$d_{50}$ ($\\mu$m)'), ('span', 'span (-)'), ('J', 'objective J')]
fig, axes = plt.subplots(1, len(columns), figsize=(19, 4.5), sharey=True)
for ax, (col, xlabel) in zip(axes, columns):
    ax.barh(labels, ranked[col], color=['C3' if ind == 0 else 'C0' for ind in range(len(ranked))])
    ax.set_xlabel(xlabel)
axes[0].invert_yaxis()
fig.tight_layout()
plt.show()

### How robust is the choice?
The KPIs do not depend on the weights, so we can re-rank the flowsheets for different priorities **without re-simulating**:

In [ ]:
weight_sets = {'balanced (1, 1, 1, 1)': (1, 1, 1, 1),
               'purity first (1, 3, 1, 1)': (1, 3, 1, 1),
               'yield first (3, 1, 1, 1)': (3, 1, 1, 1),
               'size first (1, 1, 3, 1)': (1, 1, 3, 1),
               'narrow CSD first (1, 1, 1, 3)': (1, 1, 1, 3),
               'CSD ignored (1, 1, 0, 0)': (1, 1, 0, 0),
               'size only, width ignored (1, 1, 1, 0)': (1, 1, 1, 0)}

winners = {}
for label, (w_y, w_s, w_d, w_w) in weight_sets.items():
    J = results.apply(objective, axis=1, w_yield=w_y, w_sel=w_s, w_size=w_d, w_span=w_w)
    best = results.loc[J.idxmax()]
    winners[label] = '%s → %s' % (best['reaction'], best['crystallization'])

print(pd.Series(winners, name='best flowsheet').to_string())

## Mixed-integer refinement: optimize the operating conditions too

The discrete choice above was made at **fixed** operating conditions. Each flowsheet has its own best operating conditions, and the margin between the candidates can change once they are optimized. That is why flowsheet synthesis is a mixed-integer problem.

Here we optimize the conditions of the **semibatch → batch** route, which simulates in about a second, over three continuous variables:
* reactor temperature `T_R` (lower temperature gives better selectivity, but slower reaction)
* fraction of the reaction time used for dosing B, `dose_frac`
* seed loading of the batch crystallizer, `seed_load` (%)

In [ ]:
BOUNDS = {'T_R': (313.15, 353.15), 'dose_frac': (0.2, 0.95), 'seed_load': (0.0, 10.0)}
names = list(BOUNDS)
lower = np.array([BOUNDS[name][0] for name in names])
upper = np.array([BOUNDS[name][1] for name in names])

history = []


def neg_objective(x, reactor_option='semibatch', cryst_option='batch'):
    x_clip = np.clip(x, lower, upper)
    penalty = 10 * np.sum(((x - x_clip) / (upper - lower))**2)

    ops = ...  # <-- COMPLETE THIS: copy of OPS with the decision variables replaced by x_clip
    try:
        kpis = evaluate(reactor_option, cryst_option, ops)
    except Exception as err:    # a failed simulation is just a bad point for the optimizer
        print('simulation failed at', np.round(x_clip, 3), '->', type(err).__name__)
        return 10.0
    J = objective(kpis)
    if not np.isfinite(J):      # e.g. no crystals at all
        return 10.0

    history.append({**dict(zip(names, x_clip)), **kpis, 'J': J})
    return -J + penalty


x0 = np.array([OPS['T_R'], OPS['dose_frac'], 2.0])
tic = time.perf_counter()
opt = minimize(neg_objective, x0, method='Nelder-Mead',
               options={'maxfev': 40, 'initial_simplex': [x0, x0 + [-10, 0, 0], x0 + [0, 0.15, 0], x0 + [0, 0, 3]]})
print('optimization time: %.0f s' % (time.perf_counter() - tic))

history = pd.DataFrame(history)
best_sb = history.loc[history['J'].idxmax()]
print(best_sb.round(3))

In [ ]:
base_sb = results[(results['reaction'] == 'semibatch') & (results['crystallization'] == 'batch')].iloc[0]
runner_up = ranked[~((ranked['reaction'] == 'semibatch') & (ranked['crystallization'] == 'batch'))].iloc[0]

cols = ['selectivity', 'yield', 'd50', 'span', 'J']
comparison = pd.DataFrame({
    'semibatch → batch (base)': base_sb[cols],
    'semibatch → batch (optimized)': best_sb[cols],
    'best other route (base): %s → %s' % (runner_up['reaction'], runner_up['crystallization']):
        runner_up[cols],
}).astype(float)
print(comparison.round(3).to_string())

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(-np.minimum.accumulate(-history['J'].values), 'o-', label='semibatch → batch, best so far')
ax.axhline(base_sb['J'], color='C0', ls=':', label='semibatch → batch at base conditions')
ax.axhline(runner_up['J'], color='k', ls='--', label='best other route at base conditions')
ax.set_xlabel('function evaluation')
ax.set_ylabel('best J so far')
ax.legend()
plt.show()

## Discussion and things to try

* **Look at the optimized operating conditions.** Which ones ended up at their bounds? What limits them in a real plant (solvent boiling point, batch time, seed availability)? Widen or tighten `BOUNDS` and re-run.
* **Why does the semibatch reactor win on selectivity?** Dosing keeps $C_B$ low, and $r_2/r_1 \propto C_B C_C / C_A$. What happens if you make reaction 2 first order in B (`params_f=[[1, 1], [1, 1]]`)?
* **Size vs width.** The MSMPRs give the largest $d_{50}$ but the broadest CSDs (span about 1.3, the value of an exponential distribution). Look at the re-ranking with `w_span = 0`: how much does the choice depend on the width term?
* **Yield** is limited by the solubility at `T_C` (mother liquor losses). Lower `T_C` and see how the ranking changes.
* **Missing from J**: productivity, equipment volume, cycle/start-up time and cost. Add a productivity term, e.g. kg of crystals per hour per m$^3$ of equipment.
* **Can another route catch up?** Optimize the conditions of a continuous flowsheet with `neg_objective(x, 'continuous x3', 'semibatch')`. Change `BOUNDS` to variables that matter for that route (e.g. `T_R`, `t_R`, `T_C`); `dose_frac` and `seed_load` only affect the batch routes. Each evaluation takes a few seconds, so keep `maxfev` small.
* **Two MSMPRs.** Add `'continuous x2'` to `CRYST_OPTIONS` (about 30 s per flowsheet). Does a cascade narrow the CSD enough to matter?